In [1]:
# ============================================================
# Toronto Restaurant Risk & Compliance Intelligence System
# Notebook 04: Excel Executive Summary
# Author: Sreekar Koduru
# Date: June 2026
# ============================================================

import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter

# Load cleaned datasets
CLEANED_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/data/cleaned/dinesafe_cleaned.csv'
)
SCORED_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/data/cleaned/dinesafe_risk_scored.csv'
)

df = pd.read_csv(CLEANED_PATH, parse_dates=['inspection_date'])
est = pd.read_csv(SCORED_PATH)

print("Datasets loaded ✅")
print(f"Inspections: {len(df):,} rows")
print(f"Establishments: {len(est):,} rows")

Datasets loaded ✅
Inspections: 102,766 rows
Establishments: 15,221 rows


In [3]:
# ============================================================
# STEP 2: Build Excel Executive Summary
# Why: Not all stakeholders use Power BI. City Council members,
# senior managers, and external partners often need a standalone
# Excel file they can open without any BI tool. The Excel
# summary provides the same key findings in a format everyone
# can access. It also demonstrates Excel proficiency — a core
# skill listed on every Canadian DA/BA job description.
# ============================================================

EXCEL_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/excel/dinesafe_executive_summary.xlsx'
)
os.makedirs(os.path.dirname(EXCEL_PATH), exist_ok=True)

# --- Pre-calculate all metrics ---

# KPI-01: Overall compliance rate
total_inspections = len(df)
pass_count = (df['inspection_status'] == 'Pass').sum()
conditional_count = (df['inspection_status'] == 'Conditional Pass').sum()
closed_count = (df['inspection_status'] == 'Closed').sum()
compliance_rate = round(pass_count / total_inspections * 100, 1)

# KPI-02: Avg violations per inspection
total_violations = df['severity_code'].gt(0).sum()
avg_violations = round(total_violations / total_inspections, 2)

# KPI-03: High risk establishment %
total_est = len(est)
high_risk = (est['risk_tier'] == 'High').sum()
medium_risk = (est['risk_tier'] == 'Medium').sum()
low_risk = (est['risk_tier'] == 'Low').sum()
high_risk_pct = round(high_risk / total_est * 100, 1)

# KPI-04: Repeat offender rate
repeat_offenders = est['is_repeat_offender'].sum()
repeat_rate = round(repeat_offenders / total_est * 100, 1)

# KPI-06: Critical vs minor ratio
crucial = (df['severity'] == 'C - Crucial').sum()
minor = (df['severity'] == 'M - Minor').sum()
ratio = round(crucial / minor, 3) if minor > 0 else 0

# Top violation category
top_violation = (
    df[df['deficiency_desc'].notna()]
    ['deficiency_desc']
    .value_counts()
    .index[0]
)
top_violation_count = (
    df[df['deficiency_desc'].notna()]
    ['deficiency_desc']
    .value_counts()
    .iloc[0]
)
top_violation_pct = round(
    top_violation_count /
    df['deficiency_desc'].notna().sum() * 100, 1
)

print("Metrics calculated ✅")
print(f"\nKPI-01 Compliance Rate:      {compliance_rate}%")
print(f"KPI-02 Avg Violations:       {avg_violations}")
print(f"KPI-03 High Risk Est %:      {high_risk_pct}%")
print(f"KPI-04 Repeat Offender Rate: {repeat_rate}%")
print(f"KPI-06 Crucial/Minor Ratio:  {ratio}")
print(f"Top Violation Category:      {top_violation[:50]}")

Metrics calculated ✅

KPI-01 Compliance Rate:      83.7%
KPI-02 Avg Violations:       0.59
KPI-03 High Risk Est %:      5.0%
KPI-04 Repeat Offender Rate: 22.3%
KPI-06 Crucial/Minor Ratio:  0.076
Top Violation Category:      5C. Proper Maintenance / Washing Of Rooms (Includi


In [5]:
# ============================================================
# STEP 3: Write Excel File with Formatting
# ============================================================

wb = Workbook()

# --- Define colour palette ---
DARK_BLUE  = "1E3A5F"
MID_BLUE   = "2563EB"
LIGHT_BLUE = "DBEAFE"
RED        = "DC2626"
LIGHT_RED  = "FEE2E2"
AMBER      = "D97706"
LIGHT_AMB  = "FEF3C7"
GREEN      = "16A34A"
LIGHT_GRN  = "D1FAE5"
WHITE      = "FFFFFF"
LIGHT_GREY = "F8FAFC"
MID_GREY   = "E2E8F0"

def header_cell(ws, row, col, value, bg=DARK_BLUE, fg=WHITE, size=11, bold=True):
    cell = ws.cell(row=row, column=col, value=value)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.font = Font(color=fg, bold=bold, size=size)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    return cell

def data_cell(ws, row, col, value, bg=WHITE, fg="1E293B", bold=False, align="left"):
    cell = ws.cell(row=row, column=col, value=value)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.font = Font(color=fg, bold=bold, size=10)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    return cell

def thin_border():
    thin = Side(style='thin', color="CBD5E1")
    return Border(left=thin, right=thin, top=thin, bottom=thin)

# ============================================================
# SHEET 1: KPI Scorecard
# ============================================================
ws1 = wb.active
ws1.title = "KPI Scorecard"
ws1.sheet_view.showGridLines = False
ws1.column_dimensions['A'].width = 35
ws1.column_dimensions['B'].width = 22
ws1.column_dimensions['C'].width = 22
ws1.column_dimensions['D'].width = 30

# Title
ws1.merge_cells('A1:D1')
title = ws1['A1']
title.value = "Toronto DineSafe — Executive KPI Scorecard"
title.fill = PatternFill("solid", fgColor=DARK_BLUE)
title.font = Font(color=WHITE, bold=True, size=14)
title.alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[1].height = 35

# Subtitle
ws1.merge_cells('A2:D2')
sub = ws1['A2']
sub.value = f"Analysis Period: Nov 2023 – Jun 2026  |  Data: Toronto Open Data DineSafe  |  Analyst: Sreekar Koduru"
sub.fill = PatternFill("solid", fgColor=MID_BLUE)
sub.font = Font(color=WHITE, size=10)
sub.alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[2].height = 20

# Column headers
ws1.row_dimensions[4].height = 25
for col, val in enumerate(['KPI', 'Value', 'Illustrative Target', 'Assessment'], 1):
    header_cell(ws1, 4, col, val, bg="334155", fg=WHITE)

# KPI rows
kpi_data = [
    ("KPI-01: Overall Compliance Rate",
     f"{compliance_rate}%", "≥ 85%",
     "🟡 Below target — 83.7% vs 85% goal",
     LIGHT_AMB, AMBER),

    ("KPI-02: Avg Violations Per Inspection",
     f"{avg_violations}", "≤ 1.5",
     "🟢 Within target — 0.59 violations per inspection",
     LIGHT_GRN, GREEN),

    ("KPI-03: High Risk Establishment %",
     f"{high_risk_pct}%", "≤ 10%",
     "🟢 Within target — 5% of establishments High Risk",
     LIGHT_GRN, GREEN),

    ("KPI-04: Repeat Offender Rate",
     f"{repeat_rate}%", "≤ 5%",
     "🔴 Above target — 22.3% vs 5% goal",
     LIGHT_RED, RED),

    ("KPI-06: Crucial vs Minor Ratio",
     f"{ratio}", "≤ 0.15",
     "🟢 Within target — ratio of 0.076",
     LIGHT_GRN, GREEN),

    ("KPI-07: Top Violation Category",
     "5C. Maintenance/Washing", "No category > 30%",
     "🔴 Above target — 5C accounts for 42.6% of violations",
     LIGHT_RED, RED),

    ("KPI-08: Data Coverage",
     "32 months", "Multi-year trend",
     "🟡 Limited to Nov 2023–Jun 2026 (current CSV only)",
     LIGHT_AMB, AMBER),
]

for i, (kpi, val, target, assessment, bg, accent) in enumerate(kpi_data):
    row = i + 5
    ws1.row_dimensions[row].height = 28
    row_bg = LIGHT_GREY if i % 2 == 0 else WHITE

    data_cell(ws1, row, 1, kpi, bg=row_bg, bold=True)
    data_cell(ws1, row, 2, val, bg=bg, fg=accent, bold=True, align="center")
    data_cell(ws1, row, 3, target, bg=row_bg, align="center")
    data_cell(ws1, row, 4, assessment, bg=row_bg)

    for col in range(1, 5):
        ws1.cell(row=row, column=col).border = thin_border()

# ============================================================
# SHEET 2: Risk Tier Summary
# ============================================================
ws2 = wb.create_sheet("Risk Tier Summary")
ws2.sheet_view.showGridLines = False
for col, width in zip(['A','B','C','D','E','F'], [15,20,18,18,18,20]):
    ws2.column_dimensions[col].width = width

ws2.merge_cells('A1:F1')
t = ws2['A1']
t.value = "Risk Tier Summary — All Active Establishments"
t.fill = PatternFill("solid", fgColor=DARK_BLUE)
t.font = Font(color=WHITE, bold=True, size=13)
t.alignment = Alignment(horizontal="center", vertical="center")
ws2.row_dimensions[1].height = 30

headers = ['Risk Tier', 'Establishments', '% of Total',
           'Avg Risk Score', 'Total Violations', 'Repeat Offenders']
for col, h in enumerate(headers, 1):
    header_cell(ws2, 3, col, h, bg="334155")

tier_colors = {'High': (LIGHT_RED, RED), 'Medium': (LIGHT_AMB, AMBER), 'Low': (LIGHT_GRN, GREEN)}
for i, tier in enumerate(['High', 'Medium', 'Low']):
    row = i + 4
    sub = est[est['risk_tier'] == tier]
    bg, fg = tier_colors[tier]
    ws2.row_dimensions[row].height = 25
    vals = [
        tier,
        len(sub),
        f"{len(sub)/total_est*100:.1f}%",
        f"{sub['risk_score'].mean():.1f}",
        int(sub['total_violations'].sum()),
        int(sub['is_repeat_offender'].sum()),
    ]
    aligns = ['center','center','center','center','center','center']
    for col, (v, a) in enumerate(zip(vals, aligns), 1):
        data_cell(ws2, row, col, v, bg=bg, fg=fg, bold=(col==1), align=a)
        ws2.cell(row=row, column=col).border = thin_border()

# ============================================================
# SHEET 3: Top 20 High Risk Establishments
# ============================================================
ws3 = wb.create_sheet("Top 20 High Risk")
ws3.sheet_view.showGridLines = False
for col, width in zip(['A','B','C','D','E','F','G'],
                       [5, 30, 35, 12, 12, 12, 15]):
    ws3.column_dimensions[get_column_letter(col if isinstance(col,int)
                          else ord(col)-64)].width = width

ws3.merge_cells('A1:G1')
t = ws3['A1']
t.value = "Top 20 Highest Risk Establishments — Priority Inspection List"
t.fill = PatternFill("solid", fgColor=RED)
t.font = Font(color=WHITE, bold=True, size=13)
t.alignment = Alignment(horizontal="center", vertical="center")
ws3.row_dimensions[1].height = 30

headers3 = ['#', 'Establishment', 'Address', 'Risk Score',
            'Total Violations', 'Crucial', 'Repeat Offender']
for col, h in enumerate(headers3, 1):
    header_cell(ws3, 3, col, h, bg="7F1D1D", fg=WHITE)

top20 = est.nlargest(20, 'risk_score')[
    ['est_name','address','risk_score','total_violations',
     'crucial_violations','is_repeat_offender']
].reset_index(drop=True)

for i, row_data in top20.iterrows():
    row = i + 4
    bg = LIGHT_RED if i % 2 == 0 else WHITE
    ws3.row_dimensions[row].height = 22
    vals = [
        i + 1,
        row_data['est_name'],
        row_data['address'],
        row_data['risk_score'],
        row_data['total_violations'],
        row_data['crucial_violations'],
        'Yes' if row_data['is_repeat_offender'] else 'No'
    ]
    aligns = ['center','left','left','center','center','center','center']
    for col, (v, a) in enumerate(zip(vals, aligns), 1):
        data_cell(ws3, row, col, v, bg=bg, align=a)
        ws3.cell(row=row, column=col).border = thin_border()

# ============================================================
# SHEET 4: Top Violations
# ============================================================
ws4 = wb.create_sheet("Top Violations")
ws4.sheet_view.showGridLines = False
ws4.column_dimensions['A'].width = 5
ws4.column_dimensions['B'].width = 55
ws4.column_dimensions['C'].width = 18
ws4.column_dimensions['D'].width = 18
ws4.column_dimensions['E'].width = 22

ws4.merge_cells('A1:E1')
t = ws4['A1']
t.value = "Top 15 Violation Categories by Frequency"
t.fill = PatternFill("solid", fgColor=DARK_BLUE)
t.font = Font(color=WHITE, bold=True, size=13)
t.alignment = Alignment(horizontal="center", vertical="center")
ws4.row_dimensions[1].height = 30

for col, h in enumerate(
    ['#', 'Violation Category', 'Count', '% of Total', 'Establishments Affected'], 1):
    header_cell(ws4, 3, col, h, bg="334155")

top_violations = (
    df[df['deficiency_desc'].notna()]
    .groupby('deficiency_desc')
    .agg(count=('unique_id','count'),
         establishments=('est_id','nunique'))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
total_v = top_violations['count'].sum()

for i, r in top_violations.iterrows():
    row = i + 4
    bg = LIGHT_BLUE if i % 2 == 0 else WHITE
    ws4.row_dimensions[row].height = 30
    vals = [
        i + 1,
        r['deficiency_desc'],
        r['count'],
        f"{r['count']/total_v*100:.1f}%",
        r['establishments']
    ]
    aligns = ['center','left','center','center','center']
    for col, (v, a) in enumerate(zip(vals, aligns), 1):
        c = data_cell(ws4, row, col, v, bg=bg, align=a)
        ws4.cell(row=row, column=col).border = thin_border()
        if col == 2:
            ws4.cell(row=row, column=col).alignment = Alignment(
                wrap_text=True, vertical="center")

# Save workbook
wb.save(EXCEL_PATH)
print(f"Excel Executive Summary saved ✅")
print(f"Location: excel/dinesafe_executive_summary.xlsx")
print(f"\nSheets created:")
print(f"  1. KPI Scorecard")
print(f"  2. Risk Tier Summary")
print(f"  3. Top 20 High Risk Establishments")
print(f"  4. Top Violations")

Excel Executive Summary saved ✅
Location: excel/dinesafe_executive_summary.xlsx

Sheets created:
  1. KPI Scorecard
  2. Risk Tier Summary
  3. Top 20 High Risk Establishments
  4. Top Violations


In [7]:
# ============================================================
# Export CSVs for Power BI
# Power BI Service works best with CSV uploads
# We export 3 focused datasets — one per dashboard need
# ============================================================

POWERBI_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/powerbi/'
)
os.makedirs(POWERBI_PATH, exist_ok=True)

# Dataset 1 — Inspections summary for trend analysis
inspections_summary = df[[
    'est_id', 'est_name', 'address', 'inspection_status',
    'inspection_date', 'inspection_year', 'inspection_month',
    'inspection_month_name', 'severity', 'severity_code',
    'deficiency_desc', 'type_desc', 'amount_fined',
    'latitude', 'longitude', 'is_active'
]].copy()

inspections_summary.to_csv(
    POWERBI_PATH + 'pbi_inspections.csv', index=False)
print(f"✅ pbi_inspections.csv — {len(inspections_summary):,} rows")

# Dataset 2 — Risk scores per establishment
risk_summary = est[[
    'est_id', 'est_name', 'address', 'latitude', 'longitude',
    'risk_score', 'risk_tier', 'total_violations',
    'crucial_violations', 'significant_violations',
    'minor_violations', 'is_repeat_offender',
    'times_closed', 'total_fined', 'total_inspections'
]].copy()

risk_summary.to_csv(
    POWERBI_PATH + 'pbi_risk_scores.csv', index=False)
print(f"✅ pbi_risk_scores.csv — {len(risk_summary):,} rows")

# Dataset 3 — Monthly trend for line chart
monthly_trend = df.groupby(
    ['inspection_year', 'inspection_month', 'inspection_month_name']
).agg(
    total_inspections = ('unique_id', 'count'),
    pass_count = ('inspection_status', lambda x: (x=='Pass').sum()),
    conditional_count = ('inspection_status', lambda x: (x=='Conditional Pass').sum()),
    closed_count = ('inspection_status', lambda x: (x=='Closed').sum()),
    crucial_count = ('severity_code', lambda x: (x==3).sum()),
).reset_index()

monthly_trend['pass_rate'] = (
    monthly_trend['pass_count'] /
    monthly_trend['total_inspections'] * 100
).round(1)

monthly_trend['year_month'] = (
    monthly_trend['inspection_year'].astype(str) + '-' +
    monthly_trend['inspection_month'].astype(str).str.zfill(2)
)

monthly_trend.to_csv(
    POWERBI_PATH + 'pbi_monthly_trend.csv', index=False)
print(f"✅ pbi_monthly_trend.csv — {len(monthly_trend):,} rows")

print(f"\nAll Power BI datasets exported to powerbi/ folder ✅")

✅ pbi_inspections.csv — 102,766 rows
✅ pbi_risk_scores.csv — 15,221 rows
✅ pbi_monthly_trend.csv — 32 rows

All Power BI datasets exported to powerbi/ folder ✅


In [9]:
# Get Top 20 high risk data for Power BI paste
top20_pbi = est.nlargest(20, 'risk_score')[[
    'est_name', 'address', 'latitude', 'longitude',
    'risk_score', 'risk_tier', 'total_violations',
    'crucial_violations', 'is_repeat_offender'
]].copy()

top20_pbi['is_repeat_offender'] = top20_pbi['is_repeat_offender'].map(
    {True: 'Yes', False: 'No'}
)

print(top20_pbi.to_csv(index=False))

est_name,address,latitude,longitude,risk_score,risk_tier,total_violations,crucial_violations,is_repeat_offender
Lucky Dragon Restaurant,3203 Dufferin St M6A 2T2,43.71858439,-79.45534452,100.0,High,13,2,Yes
Omni Noodle,4271 Sheppard Ave E Unit-3-4 M1S 4G4,43.78553795,-79.2756602,100.0,High,12,3,Yes
Kerala Kafe Indian Cuisine,3258 Lawrence Ave E Unit-B M1P 2P5,43.75731349,-79.23860168,100.0,High,16,3,Yes
August 8 Eglinton,10 Lebovic Ave Unit-8 M1L 4V9,43.72433492,-79.2908201,100.0,High,11,1,Yes
Kai Wei Supermarket,253 Spadina Ave M5T 2E3,43.651886,-79.397514,100.0,High,11,2,Yes
Sushi Cafe Bon Gung,109 Mc Caul St Unit-40 M5T 3K5,43.65413,-79.3911,100.0,High,14,1,Yes
The Host Fine Indian Cuisine,87 Elm St M5G 0A8,43.65672391,-79.38641158,100.0,High,12,2,Yes
Pizza Pizza,340 Front St W M5V 3W7,43.643738,-79.39148,100.0,High,13,3,Yes
The Bombay,259 Queen St W M5V 1Z4,43.65008732,-79.38900904,100.0,High,14,2,Yes
Oeb Breakfast Co,125 East Liberty St M6K 3K4,43.63835496,-79.41618154,100.0,High,9